In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure numeric values where expected
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Aggregate metric.csv by cmdb_id and kpi_name (global stats on full series)
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_agg = grp.agg(
    count='count',
    min_value='min',
    max_value='max',
    mean_value='mean',
    median_value=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    p99=lambda x: x.quantile(0.99)
).reset_index()

# Keep the full aggregated table in memory (metric_agg) for later use.
# Return only top 50 rows ordered by count descending (compact)
metric_top50 = metric_agg.sort_values('count', ascending=False).head(50)

# 2) From trace.csv: distinct trace_name and distinct cmdb_id (up to 50 each)
trace_names = (trace_df['trace_name']
               .dropna()
               .value_counts()
               .reset_index()
               .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
               .head(50))

trace_cmdbs = (trace_df['cmdb_id']
               .dropna()
               .value_counts()
               .reset_index()
               .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
               .head(50))

# 3) From log.csv: distinct log_name and distinct cmdb_id (up to 50 each)
log_names = (log_df['log_name']
             .dropna()
             .value_counts()
             .reset_index()
             .rename(columns={'index': 'log_name', 'log_name': 'count'})
             .head(50))

log_cmdbs = (log_df['cmdb_id']
             .dropna()
             .value_counts()
             .reset_index()
             .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
             .head(50))

# 4) From error_logs.csv: distinct cmdb_id and up to 20 sample messages (timestamp, cmdb_id, message)
error_cmdbs = (error_df['cmdb_id']
               .dropna()
               .value_counts()
               .reset_index()
               .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'}))

# Sample up to 20 messages for context (sorted by timestamp ascending)
if 'message' in error_df.columns:
    error_samples = error_df[['timestamp', 'dt', 'cmdb_id', 'message']].sort_values('timestamp').head(20)
else:
    # Fallback if message missing
    error_samples = error_df[['timestamp', 'dt', 'cmdb_id']].sort_values('timestamp').head(20)

# Final compact outputs (variables shown as requested)
metric_top50, trace_names, trace_cmdbs, log_names, log_cmdbs, error_cmdbs, error_samples

```
Out[1]:
```
metric_agg_rows = metric_agg.shape[0]
top50_rows = metric_top50.shape[0]
most_common_count = int(metric_top50['count'].mode()[0]) if 'count' in metric_top50 else None
sample_metrics = ", ".join(
    metric_top50[['cmdb_id', 'kpi_name', 'count']].head(3)
    .apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']} (count={int(r['count'])})", axis=1)
    .tolist()
)

trace_top_names = ", ".join(
    trace_names.head(3).apply(lambda r: f"{r['trace_name']} (count={int(r['count'])})", axis=1).tolist()
)
trace_top_cmdbs = ", ".join(
    trace_cmdbs.head(6).apply(lambda r: f"{r['cmdb_id']} (count={int(r['count'])})", axis=1).tolist()
)

log_top_names = ", ".join(
    log_names.head(3).apply(lambda r: f"{r['log_name']} (count={int(r['count'])})", axis=1).tolist()
)
log_top_cmdbs = ", ".join(
    log_cmdbs.head(6).apply(lambda r: f"{r['cmdb_id']} (count={int(r['count'])})", axis=1).tolist()
)

error_cmdbs_count = error_cmdbs.shape[0] if 'error_cmdbs' in globals() else 0
error_samples_count = error_samples.shape[0] if 'error_samples' in globals() else 0
error_note = "no error log samples found" if error_samples_count == 0 else f"{error_samples_count} sample error log rows available"

summary = (
    "Summary of telemetry files (no incident-time filtering):\n\n"
    f"1) metric.csv aggregations: computed global stats (count, min, max, mean, median, P90, P95, P99) grouped by (cmdb_id, kpi_name). "
    f"Full aggregated table stored in variable `metric_agg` ({metric_agg_rows} rows). Top results returned in `metric_top50` ({top50_rows} rows), "
    f"most common count in the top 50 = {most_common_count}. Sample top metrics: {sample_metrics}.\n\n"
    f"2) trace.csv distinct names and cmdb_ids (up to 50 each): top trace names: {trace_top_names}. "
    f"Top trace cmdb_ids: {trace_top_cmdbs}.\n\n"
    f"3) log.csv distinct names and cmdb_ids (up to 50 each): top log names: {log_top_names}. "
    f"Top log cmdb_ids: {log_top_cmdbs}.\n\n"
    f"4) error_logs.csv: distinct cmdb_id rows = {error_cmdbs_count}; {error_note}. "
    "If empty, the error_logs.csv file contains no usable (timestamp, cmdb_id, message) samples in the dataset provided.\n\n"
    "Notes: timestamps were converted to Asia/Shanghai (UTC+8). Use `metric_agg` for further metric-level analysis."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     min_value     max_value    mean_value  median_value           p90           p95           p99
1282  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
88        IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.175232e+06  6.177038e+06  6.176135e+06  6.176135e+06  6.176857e+06  6.176947e+06  6.177020e+06
1289  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.700000e+01  7.500000e+01  7.009677e+01  6.900000e+01  7.400000e+01  7.500000e+01  7.500000e+01
1281  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
...        ...                                                ...    ...           ...           ...           ...           ...           ...           ...           ...
104       IG02                             OSLinux-CPU_CPU_CPUWio     31  8.300000e-03  1.026302e+00  1.513443e-01  3.740000e-02  3.366580e-01  5.713295e-01  9.028814e-01
1399  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
1384  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.082270e+05  2.083150e+05  2.082737e+05  2.082720e+05  2.083150e+05  2.083150e+05  2.083150e+05
1385  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1386  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 10 columns],                                 count  count
0            trace.self.duration_mean    250
1             trace.self.duration_p95    250
2                trace.self.row_count    250
3             trace.to_MG02.row_count    102
4         trace.to_MG02.duration_mean    102
..                                ...    ...
45   trace.from_Tomcat02.duration_p95     36
46  trace.from_Tomcat02.duration_mean     36
47      trace.from_Tomcat02.row_count     36
48   trace.from_Tomcat03.duration_p95     34
49  trace.from_Tomcat03.duration_mean     34

[50 rows x 2 columns],        count  count
0       MG01    537
1       MG02    531
2   Tomcat01    318
3       IG01    312
4       IG02    297
5   Tomcat04    288
6   Tomcat02    285
7   Tomcat03    276
8   dockerB1    246
9   dockerB2    219
10  dockerA1    117
11  dockerA2     84,               count  count
0   log.error_count    121
1     log.row_count    121
2  log.gc_pause_max      8,       count  count
0  Tomcat01     52
1  Tomcat04     50
2  apache02     50
3  Tomcat03     48
4  Tomcat02     46
5   Mysql01      4, Empty DataFrame
Columns: [count, count]
Index: [], Empty DataFrame
Columns: [timestamp, dt, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse variables from previous kernel state: metric_df, metric_agg (if present)
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is available and has dt column with Asia/Shanghai timezone
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# create/ensure timezone-aware datetime column 'dt' in Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # if exists but not tz-aware, convert properly
    if metric_df['dt'].dtype == 'int64' or metric_df['dt'].dtype == 'float64':
        metric_df['dt'] = pd.to_datetime(metric_df['dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # try to ensure tz
        if metric_df['dt'].dt.tz is None:
            metric_df['dt'] = metric_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['dt'] = metric_df['dt'].dt.tz_convert(tz)

# 1) Compute global thresholds P95 and P05 per (cmdb_id, kpi_name) using the full series
thresholds = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p05=lambda x: x.quantile(0.05)
).reset_index()

# 2) Join thresholds back to every metric row
metric_with_thresh = metric_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Define incident window in Asia/Shanghai (UTC+8)
start_ts = tz.localize(pd.Timestamp('2021-03-06 04:30:00'))
end_ts = tz.localize(pd.Timestamp('2021-03-06 05:00:00'))

# 4) Filter rows to the incident window
window_df = metric_with_thresh[(metric_with_thresh['dt'] >= start_ts) & (metric_with_thresh['dt'] <= end_ts)].copy()

# 5) Mark anomalies: value >= p95 (high) OR value <= p05 (low)
# handle NaNs safely: if p95/p05 are NaN or value is NaN, anomaly=False
window_df['anomaly'] = False
valid_mask = window_df['value'].notna() & window_df['p95'].notna() & window_df['p05'].notna()
window_df.loc[valid_mask, 'anomaly'] = (window_df.loc[valid_mask, 'value'] >= window_df.loc[valid_mask, 'p95']) | \
                                      (window_df.loc[valid_mask, 'value'] <= window_df.loc[valid_mask, 'p05'])

# 6) Store full anomalies table for later use
metric_anomalies = window_df[window_df['anomaly']].copy()

# 7) Aggregate anomalies per cmdb_id:
# - anomaly_count
# - earliest_anomaly_ts (Asia/Shanghai)
# - top 3 anomalous kpi_name with their anomaly counts
anomaly_counts = metric_anomalies.groupby('cmdb_id').size().reset_index(name='anomaly_count')
earliest_ts = metric_anomalies.groupby('cmdb_id')['dt'].min().reset_index(name='earliest_anomaly_dt')

# top KPIs per cmdb_id
kp_counts = metric_anomalies.groupby(['cmdb_id', 'kpi_name']).size().reset_index(name='kp_anom_count')
kp_counts = kp_counts.sort_values(['cmdb_id', 'kp_anom_count'], ascending=[True, False])

# take top 3 per cmdb_id and summarize into a compact string
topk = kp_counts.groupby('cmdb_id').head(3)
topk_summary = topk.groupby('cmdb_id').apply(
    lambda df: ', '.join(df.apply(lambda r: f"{r['kpi_name']}({int(r['kp_anom_count'])})", axis=1))
).reset_index(name='top_kpis_summary')

# join aggregates
anomaly_summary = anomaly_counts.merge(earliest_ts, on='cmdb_id', how='left').merge(topk_summary, on='cmdb_id', how='left')

# format earliest timestamp as compact string in Asia/Shanghai
anomaly_summary['earliest_anomaly_ts'] = anomaly_summary['earliest_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
anomaly_summary = anomaly_summary.drop(columns=['earliest_anomaly_dt'])

# 8) Return compact summary table limited to top 20 cmdb_id ordered by anomaly_count desc
anomaly_summary = anomaly_summary.sort_values('anomaly_count', ascending=False).head(20)

# Final displayed variables: anomaly_summary (compact) and metric_anomalies.shape (size of full anomalies table stored)
anomaly_summary, metric_anomalies.shape

```
Out[2]:
```
summary = (
    "Anomaly summary (metrics) for window 2021-03-06 04:30–05:00 Asia/Shanghai:\n\n"
    "- Total anomaly rows detected: 18917 (stored in variable `metric_anomalies`).\n"
    "- Top affected hosts by anomaly count (earliest anomaly shown ~2021-03-06 04:30:00 CST):\n"
    "  1) Mysql01: 3156   2) Mysql02: 2788   3) Redis01: 1606   4) Redis02: 1593\n"
    "  5) Tomcat02: 1253  6) Tomcat03: 1250  7) Tomcat04: 1225  8) Tomcat01: 1152\n"
    "  9) IG02: 1006     10) IG01: 970     11) apache02: 898    12) MG02: 739 (top 20 available in `anomaly_summary`).\n\n"
    "- Dominant anomalous KPI patterns observed:\n"
    "  * Many DBs/infra (Mysql*, Redis*, apache*) show OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT as the top anomalous metric.\n"
    "  * Tomcat instances mainly show Tomcat-Requests ... _ErrorCount as top anomalous KPI.\n"
    "  * IG/MG nodes show JVM-Memory_..._HeapMemoryMax related anomalies.\n\n"
    "- Notes and recommended next steps:\n"
    "  * Network/TCP CLOSE-WAIT spikes on DB/Redis/apache hosts warrant investigating network/socket handling and connection cleanup.\n"
    "  * Tomcat error-count anomalies suggest checking application logs and request handling for errors during the window.\n"
    "  * JVM heap maxima on IG/MG should be inspected (heap pressure / GC / OOM risk).\n\n"
    "Variables for follow-up: `metric_anomalies` (full anomaly rows, shape shown earlier) and `anomaly_summary` (compact per-cmdb_id summary, top 20)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  anomaly_count                                   top_kpis_summary      earliest_anomaly_ts
4         Mysql01           3156  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...  2021-03-06 04:30:00 CST
5         Mysql02           2788  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...  2021-03-06 04:30:00 CST
6         Redis01           1606  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...  2021-03-06 04:30:00 CST
7         Redis02           1593  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...  2021-03-06 04:30:00 CST
18       Tomcat02           1253  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-06 04:30:00 CST
19       Tomcat03           1250  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-06 04:30:00 CST
20       Tomcat04           1225  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-06 04:30:00 CST
17       Tomcat01           1152  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-06 04:30:00 CST
1            IG02           1006  JVM-Memory_7778_JVM_Memory_HeapMemoryMax(31), ...  2021-03-06 04:30:00 CST
0            IG01            970  JVM-Memory_7778_JVM_Memory_HeapMemoryMax(31), ...  2021-03-06 04:30:00 CST
22       apache02            898  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...  2021-03-06 04:30:00 CST
3            MG02            739  JVM-Memory_7779_JVM_Memory_HeapMemoryMax(31), ...  2021-03-06 04:30:00 CST
2            MG01            724  JVM-Memory_7779_JVM_Memory_HeapMemoryMax(31), ...  2021-03-06 04:30:00 CST
21       apache01            275  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies(...  2021-03-06 04:30:00 CST
24       dockerA2             49  Container-DOCKER_CONTAINER_2fa7eaebac26--bcou-...  2021-03-06 04:34:00 CST
8    ServiceTest1             40                 app.rr(16), app.sr(16), app.cnt(6)  2021-03-06 04:32:00 CST
9   ServiceTest10             26                 app.rr(10), app.sr(10), app.cnt(4)  2021-03-06 04:33:00 CST
13   ServiceTest6             24                   app.rr(9), app.sr(9), app.cnt(4)  2021-03-06 04:37:00 CST
11   ServiceTest4             23                   app.rr(9), app.sr(9), app.cnt(3)  2021-03-06 04:37:00 CST
14   ServiceTest7             23                   app.rr(8), app.sr(8), app.cnt(5)  2021-03-06 04:37:00 CST, (18917, 8))```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# reuse trace_df if present; otherwise load
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# ensure dt column timezone-aware in Asia/Shanghai
if 'dt' not in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize existing dt column to tz-aware Asia/Shanghai
    if pd.api.types.is_numeric_dtype(trace_df['dt']):
        trace_df['dt'] = pd.to_datetime(trace_df['dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if trace_df['dt'].dt.tz is None:
            trace_df['dt'] = trace_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            trace_df['dt'] = trace_df['dt'].dt.tz_convert(tz)

# 1) Global statistics per trace_name over full series (count, P05, P50, P90, P95)
trace_agg = trace_df.groupby('trace_name')['value'].agg(
    count='count',
    p05=lambda x: x.quantile(0.05),
    p50=lambda x: x.quantile(0.50),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# store counts-only top50 (per request in step 1) in a variable (not printed here)
trace_counts_top50 = trace_agg[['trace_name', 'count']].sort_values('count', ascending=False).head(50)

# 2) Filter trace rows to incident window 2021-03-06 04:30 to 2021-03-06 05:00 Asia/Shanghai
start_ts = tz.localize(pd.Timestamp('2021-03-06 04:30:00'))
end_ts = tz.localize(pd.Timestamp('2021-03-06 05:00:00'))
trace_window = trace_df[(trace_df['dt'] >= start_ts) & (trace_df['dt'] <= end_ts)].copy()

# 3) Join filtered rows with global P95 and P05, mark anomalies
trace_window = trace_window.merge(trace_agg[['trace_name', 'p95', 'p05']], on='trace_name', how='left')
valid_mask = trace_window['value'].notna() & trace_window['p95'].notna() & trace_window['p05'].notna()
trace_window['anomaly'] = False
trace_window.loc[valid_mask, 'anomaly'] = (trace_window.loc[valid_mask, 'value'] >= trace_window.loc[valid_mask, 'p95']) | \
                                         (trace_window.loc[valid_mask, 'value'] <= trace_window.loc[valid_mask, 'p05'])

# save full set of anomalous trace rows
trace_anomalies = trace_window[trace_window['anomaly']].copy()

# 4) Aggregate trace_anomalies by trace_name:
# anomaly_count, earliest_anomaly_timestamp (Asia/Shanghai), sample component token (to_X/from_X/self)
if not trace_anomalies.empty:
    agg_count = trace_anomalies.groupby('trace_name').size().reset_index(name='anomaly_count')
    agg_earliest = trace_anomalies.groupby('trace_name')['dt'].min().reset_index(name='earliest_dt')
    agg = agg_count.merge(agg_earliest, on='trace_name', how='left')

    # function to extract sample token from trace_name
    def extract_token(name):
        # look for to_X or from_X
        m = re.search(r'\b(to_[A-Za-z0-9]+|from_[A-Za-z0-9]+)\b', name)
        if m:
            return m.group(1)
        if 'self' in name:
            return 'self'
        return ''

    # sample token per trace_name (use the trace_name itself since tokens are embedded there)
    agg['sample_component_token'] = agg['trace_name'].apply(extract_token)

    # format earliest timestamp
    agg['earliest_anomaly_ts'] = agg['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_summary = agg[['trace_name', 'anomaly_count', 'earliest_anomaly_ts', 'sample_component_token']]\
                    .sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    # no anomalies found in window -> empty summary
    trace_summary = pd.DataFrame(columns=['trace_name', 'anomaly_count', 'earliest_anomaly_ts', 'sample_component_token'])

# Display only the compact summary table as requested
trace_summary

```
Out[3]:
```
summary = (
    "Trace anomalies (window 2021-03-06 04:30–05:00 Asia/Shanghai):\n\n"
    "Top anomalous trace names (count, earliest timestamp, token):\n"
    "1) trace.from_IG01.row_count — 49 anomalies, earliest 2021-03-06 04:32:00 CST (from_IG01)\n"
    "2) trace.self.duration_p95 — 42 anomalies, earliest 2021-03-06 04:32:00 CST (self)\n"
    "3) trace.to_MG02.row_count — 41 anomalies, earliest 2021-03-06 04:32:00 CST (to_MG02)\n"
    "4) trace.to_MG01.row_count — 36 anomalies, earliest 2021-03-06 04:35:00 CST (to_MG01)\n"
    "5) trace.from_IG02.row_count — 32 anomalies, earliest 2021-03-06 04:33:00 CST (from_IG02)\n\n"
    "Key observations:\n"
    "- Many anomalies are in 'row_count' traces for directional tokens (to_X / from_X) involving IG and MG, indicating unusual traffic/throughput between gateway/middleware and services.\n"
    "- 'self' traces (duration and row_count) also show up high, suggesting service-local latency or processing changes.\n"
    "- Affected components in the top list include IG01/IG02, MG01/MG02, and various Tomcat targets (Tomcat01–04).\n\n"
    "Recommended next steps:\n"
    "- Inspect service call chains and span details for the listed trace names (start with IG and MG interactions).\n"
    "- Correlate with application logs and metrics on the same hosts to determine if increased load, latency, or downstream failures caused the anomalies.\n"
    "- Prioritize traces with high anomaly counts (e.g., trace.from_IG01.row_count, trace.self.duration_p95) for deeper tracing and root-cause analysis.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  anomaly_count      earliest_anomaly_ts sample_component_token
0       trace.from_IG01.row_count             49  2021-03-06 04:32:00 CST              from_IG01
1         trace.self.duration_p95             42  2021-03-06 04:32:00 CST                   self
2         trace.to_MG02.row_count             41  2021-03-06 04:32:00 CST                to_MG02
3         trace.to_MG01.row_count             36  2021-03-06 04:35:00 CST                to_MG01
4       trace.from_IG02.row_count             32  2021-03-06 04:33:00 CST              from_IG02
5            trace.self.row_count             32  2021-03-06 04:32:00 CST                   self
6        trace.self.duration_mean             26  2021-03-06 04:33:00 CST                   self
7     trace.to_Tomcat01.row_count             22  2021-03-06 04:33:00 CST            to_Tomcat01
8     trace.to_Tomcat02.row_count             20  2021-03-06 04:33:00 CST            to_Tomcat02
9   trace.from_Tomcat02.row_count             19  2021-03-06 04:33:00 CST          from_Tomcat02
10    trace.to_Tomcat04.row_count             18  2021-03-06 04:32:00 CST            to_Tomcat04
11  trace.from_Tomcat01.row_count             18  2021-03-06 04:33:00 CST          from_Tomcat01
12      trace.from_MG01.row_count             17  2021-03-06 04:35:00 CST              from_MG01
13    trace.to_Tomcat03.row_count             16  2021-03-06 04:35:00 CST            to_Tomcat03
14      trace.from_MG02.row_count             15  2021-03-06 04:32:00 CST              from_MG02
15  trace.from_Tomcat04.row_count             15  2021-03-06 04:32:00 CST          from_Tomcat04
16    trace.to_MG01.duration_mean             14  2021-03-06 04:38:00 CST                to_MG01
17     trace.to_MG01.duration_p95             13  2021-03-06 04:38:00 CST                to_MG01
18    trace.to_MG02.duration_mean             13  2021-03-06 04:37:00 CST                to_MG02
19     trace.to_MG02.duration_p95             12  2021-03-06 04:37:00 CST                to_MG02```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_anomalies is available
if 'metric_anomalies' not in globals():
    raise RuntimeError("metric_anomalies variable not found in kernel. Please run anomaly detection step first.")

# Work on a copy and ensure required columns exist
ma = metric_anomalies.copy()
# Ensure timestamp is integer seconds and dt is timezone-aware Asia/Shanghai
if 'timestamp' not in ma.columns:
    ma['timestamp'] = ma['dt'].astype('int64') // 10**9  # fallback
ma['timestamp'] = ma['timestamp'].astype(int)

if 'dt' not in ma.columns:
    ma['dt'] = pd.to_datetime(ma['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize dt to tz-aware Asia/Shanghai
    if pd.api.types.is_numeric_dtype(ma['dt']):
        ma['dt'] = pd.to_datetime(ma['dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if ma['dt'].dt.tz is None:
            ma['dt'] = ma['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            ma['dt'] = ma['dt'].dt.tz_convert(tz)

# Sort for deterministic processing
ma = ma.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).reset_index(drop=True)

faults = []

# Group by cmdb_id and kpi_name to find consecutive-minute sequences
for (cmdb_id, kpi), grp in ma.groupby(['cmdb_id', 'kpi_name'], sort=False):
    g = grp.sort_values('timestamp').copy()
    # compute diff in seconds between successive anomaly timestamps
    g['prev_ts'] = g['timestamp'].shift(1)
    g['diff'] = (g['timestamp'] - g['prev_ts']).fillna(-999).astype(int)
    # start new sequence where diff != 60 (first row will start a new seq)
    g['seq_id'] = (g['diff'] != 60).cumsum()
    # aggregate each sequence
    for seq, sgrp in g.groupby('seq_id', sort=True):
        start_ts = int(sgrp['timestamp'].iloc[0])
        end_ts = int(sgrp['timestamp'].iloc[-1])
        streak_len = int(sgrp.shape[0])
        max_value = float(sgrp['value'].max())
        min_value = float(sgrp['value'].min())
        # determine threshold_type: 'high' if any value >= p95 else 'low'
        any_high = (sgrp['value'] >= sgrp['p95']).any()
        threshold_type = 'high' if any_high else 'low'
        # threshold_value: take the group's p95 or p05 (they should be same across group)
        if threshold_type == 'high':
            threshold_value = float(sgrp['p95'].iloc[0])
            # compute deviation using max_value
            if threshold_value == 0:
                max_deviation_pct = np.nan
            else:
                max_deviation_pct = (max_value - threshold_value) / threshold_value * 100.0
        else:
            threshold_value = float(sgrp['p05'].iloc[0])
            # compute deviation using min_value
            if threshold_value == 0:
                max_deviation_pct = np.nan
            else:
                max_deviation_pct = (threshold_value - min_value) / threshold_value * 100.0

        faults.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'start_timestamp': start_ts,
            'end_timestamp': end_ts,
            'streak_length': streak_len,
            'max_value': max_value,
            'min_value': min_value,
            'threshold_type': threshold_type,
            'threshold_value': threshold_value,
            'max_deviation_pct': float(max_deviation_pct) if np.isfinite(max_deviation_pct) else np.nan
        })

# Create DataFrame of faults and store in kernel variable metric_faults
metric_faults = pd.DataFrame(faults)

# Add human-readable datetime strings in Asia/Shanghai for start/end
if not metric_faults.empty:
    metric_faults['start_dt'] = pd.to_datetime(metric_faults['start_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_faults['end_dt'] = pd.to_datetime(metric_faults['end_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_faults['start_ts_str'] = metric_faults['start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_faults['end_ts_str'] = metric_faults['end_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    # ensure columns exist even if empty
    metric_faults = metric_faults.reindex(columns=[
        'cmdb_id','kpi_name','start_timestamp','end_timestamp','streak_length',
        'max_value','min_value','threshold_type','threshold_value','max_deviation_pct',
        'start_dt','end_dt','start_ts_str','end_ts_str'
    ])

# Prepare compact summary: top 20 faults ordered by streak_length desc then max_deviation_pct desc
metric_faults_summary = metric_faults.sort_values(
    by=['streak_length', 'max_deviation_pct'], ascending=[False, False]
).head(20)

# Select and format required output columns
metric_faults_summary = metric_faults_summary[[
    'cmdb_id',
    'kpi_name',
    'start_ts_str',
    'end_ts_str',
    'streak_length',
    'threshold_type',
    'threshold_value',
    'max_deviation_pct'
]].rename(columns={
    'start_ts_str': 'start_timestamp',
    'end_ts_str': 'end_timestamp'
}).reset_index(drop=True)

# Display compact summary only
metric_faults_summary

```
Out[4]:
```
Summary of consecutive-minute faults (window 2021-03-06 04:30–05:00 Asia/Shanghai):

- Many metrics showed continuous anomalies across the entire incident window (streak_length = 31 minutes) for multiple hosts and KPIs — i.e., anomalies at every minute from 04:30 to 05:00.
- Affected hosts include Redis02, IG01, IG02, MG01, MG02, Mysql01, Redis01, Tomcat01 and others. These faults cover both infra/process metrics and JVM/Tomcat metrics.
- Top KPI patterns observed in these faults:
  - OSLinux process metrics (e.g., zabbix agent process, PROC PCount/NoZombies) repeatedly flagged as high.
  - JVM-Memory_*_HeapMemoryMax and related JVM memory KPIs on IG/MG nodes flagged as high.
  - Tomcat Request/ErrorCount metrics on Tomcat instances flagged as high for the full window.
- Threshold type: all listed faults are "high" anomalies (value >= P95) — i.e., values exceeded their global P95 thresholds.
- Deviation magnitude: most sequences report max_deviation_pct = 0.0 (values at or equal to the threshold), with a notable exception: Redis02 OSLinux process zabbix agentd had max_deviation_pct ~25%, indicating a substantially higher value above its threshold.
- Interpretation / recommended next steps:
  - These are persistent faults (not transient spikes). Prioritize investigation of hosts with largest operational impact (e.g., Mysql01/Mysql02, Redis01/Redis02, Tomcat nodes, IG/MG).
  - For JVM-related faults: inspect heap usage, recent GC activity, and application logs for memory pressure or misconfiguration.
  - For process/zabbix-related faults: inspect process counts, zabbix agent behavior, and any automated process restarts or zombie processes.
  - For Tomcat error-counts: review application logs and request traces during the window to find root cause of repeated request errors.
  - Correlate these faults with the trace anomalies and error logs already collected to narrow root cause (network/socket issues, connection handling, or application errors).
- Data locations: full faults table saved in metric_faults; a compact top-20 listing (as shown) identifies the highest-priority fault sequences.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name          start_timestamp            end_timestamp  streak_length threshold_type  threshold_value  max_deviation_pct
0    Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     4.000000e-01               25.0
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     2.040136e+09                0.0
2       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     1.000000e+00                0.0
3       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     3.000000e-01                0.0
4       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     3.000000e+00                0.0
5       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     2.040136e+09                0.0
6       IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     1.000000e+00                0.0
7       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     3.000000e-01                0.0
8       IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     3.000000e+00                0.0
9       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     2.040136e+09                0.0
10      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     5.063234e+08                0.0
11      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     2.040136e+09                0.0
12      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     9.472872e+08                0.0
13   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     4.000000e-01                0.0
14   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     7.000000e+00                0.0
15   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     1.000000e+00                0.0
16   Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     4.000000e-01                0.0
17   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     4.000000e+00                0.0
18   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     4.000000e+00                0.0
19  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-06 04:30:00 CST  2021-03-06 05:00:00 CST             31           high     2.300000e+01                0.0```
```

In [ ]:
# In[5]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing kernel variables: metric_faults, trace_anomalies
# ensure they exist
if 'metric_faults' not in globals():
    raise RuntimeError("metric_faults not found in kernel. Run previous fault-detection step first.")
if 'trace_anomalies' not in globals():
    raise RuntimeError("trace_anomalies not found in kernel. Run previous trace anomaly step first.")

# Work on copies to avoid modifying originals unintentionally
tf = trace_anomalies.copy()
mf = metric_faults.copy()

# Ensure trace dt column is timezone-aware Asia/Shanghai
if 'dt' in tf.columns:
    if pd.api.types.is_numeric_dtype(tf['dt']):
        tf['dt'] = pd.to_datetime(tf['dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if tf['dt'].dt.tz is None:
            tf['dt'] = tf['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            tf['dt'] = tf['dt'].dt.tz_convert(tz)
else:
    # fallback from timestamp if present
    if 'timestamp' in tf.columns:
        tf['dt'] = pd.to_datetime(tf['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        tf['dt'] = pd.NaT

# 1) Extract component token from trace_name
def extract_component_token(trace_name, row_cmdb):
    if not isinstance(trace_name, str):
        return ''
    m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)\b', trace_name)
    if m:
        return m.group(1)
    if 'self' in trace_name:
        return row_cmdb if pd.notna(row_cmdb) else 'self'
    return ''

tf['trace_component'] = tf.apply(lambda r: extract_component_token(r.get('trace_name',''), r.get('cmdb_id', '')), axis=1)

# 2) Build per-component summary from metric_faults (group by cmdb_id)
# Ensure start_dt exists and is tz-aware
if 'start_dt' in mf.columns:
    if pd.api.types.is_numeric_dtype(mf['start_dt']):
        mf['start_dt'] = pd.to_datetime(mf['start_dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if mf['start_dt'].dt.tz is None:
            mf['start_dt'] = mf['start_dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            mf['start_dt'] = mf['start_dt'].dt.tz_convert(tz)
else:
    # fallback to numeric start_timestamp if present
    if 'start_timestamp' in mf.columns:
        mf['start_dt'] = pd.to_datetime(mf['start_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        mf['start_dt'] = pd.NaT

metric_group = mf.groupby('cmdb_id').agg(
    metric_fault_count=('kpi_name', 'count'),
    earliest_metric_fault_start_dt=('start_dt', 'min'),
    max_metric_streak=('streak_length', 'max'),
    worst_max_deviation_pct=('max_deviation_pct', 'max')
).reset_index().rename(columns={'cmdb_id':'component'})

# format earliest_metric_fault_start
metric_group['earliest_metric_fault_start'] = metric_group['earliest_metric_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_group.loc[metric_group['earliest_metric_fault_start_dt'].isna(), 'earliest_metric_fault_start'] = pd.NA
metric_group = metric_group.drop(columns=['earliest_metric_fault_start_dt'])

# 2b) Build trace summary per extracted component
trace_group = tf[tf['trace_component'] != ''].groupby('trace_component').agg(
    trace_anomaly_count=('trace_component', 'count'),
    earliest_trace_anomaly_dt=('dt', 'min')
).reset_index().rename(columns={'trace_component':'component'})

trace_group['earliest_trace_anomaly_ts'] = trace_group['earliest_trace_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_group.loc[trace_group['earliest_trace_anomaly_dt'].isna(), 'earliest_trace_anomaly_ts'] = pd.NA
trace_group = trace_group.drop(columns=['earliest_trace_anomaly_dt'])

# 2c) Outer merge metric_group and trace_group to include components present in either
comp_summary = metric_group.merge(trace_group, on='component', how='outer')

# Fill NA counts with 0 for readability
comp_summary['metric_fault_count'] = comp_summary['metric_fault_count'].fillna(0).astype(int)
comp_summary['trace_anomaly_count'] = comp_summary['trace_anomaly_count'].fillna(0).astype(int)

# Ensure numeric columns for sorting
comp_summary['max_metric_streak'] = comp_summary['max_metric_streak'].fillna(0).astype(int)
comp_summary['worst_max_deviation_pct'] = comp_summary['worst_max_deviation_pct'].replace([pd.NA, None], pd.NA).astype(float)

# 3) Order by trace_anomaly_count desc then metric_fault_count desc and limit to top 20
component_trace_metric_summary = comp_summary.sort_values(
    by=['trace_anomaly_count', 'metric_fault_count'], ascending=[False, False]
).head(20).reset_index(drop=True)

# Select and order columns as requested
component_trace_metric_summary = component_trace_metric_summary[[
    'component',
    'metric_fault_count',
    'earliest_metric_fault_start',
    'max_metric_streak',
    'worst_max_deviation_pct',
    'trace_anomaly_count',
    'earliest_trace_anomaly_ts'
]]

# Display the compact summary (top 20)
component_trace_metric_summary

```
Out[5]:
```
Summary of combined metric faults + trace anomalies (top 20 components):

Key findings
- Components with both large trace anomaly activity and many metric fault sequences (high priority):
  - MG02: 101 trace anomalies, 594 metric faults; earliest trace anomaly 2021-03-06 04:32, earliest metric fault 04:30, max fault streak 31 min, worst deviation 100%.
  - MG01: 98 trace anomalies, 571 metric faults; earliest trace 04:35, worst deviation ~82%.
  - IG01: 87 trace anomalies, 693 metric faults; earliest trace 04:32, worst deviation ~233%.
  - Tomcat01: 64 trace anomalies, 629 metric faults; earliest trace 04:33, worst deviation 100%.
  - IG02: 63 trace anomalies, 721 metric faults; earliest trace 04:33, worst deviation ~233%.
  - Tomcat02: 61 trace anomalies, 629 metric faults; earliest trace 04:33, worst deviation 100%.

- Components with many metric faults but no matching trace anomalies in the window (consider investigating infra-level issues separate from tracing):
  - Mysql01: 2053 metric faults, 0 trace anomalies.
  - Mysql02: 1835 metric faults, 0 trace anomalies.
  - Redis01 / Redis02, apache01 / apache02 also show many metric faults but trace_anomaly_count = 0 in this extraction.

- Components with trace anomalies but few metric faults (possible localized trace-level issues):
  - dockerB2, dockerB1, dockerA1, dockerA2 — have non-zero trace anomaly counts but low metric_fault_count and short metric streaks.

Interpretation and next steps
- Prioritize investigation for components with both high trace_anomaly_count and high metric_fault_count (MG01/MG02, IG01/IG02, Tomcat nodes). These show correlated tracing and persistent metric faults (many metric fault sequences of length 31 minutes).
- For components with high metric faults but no trace anomalies (Mysql*, Redis*, apache*), focus on infra processes, network/socket metrics and DB/daemon health; tracing may not capture these issues.
- Use the stored summary variable component_trace_metric_summary for follow-up (it contains the compact joined table), and use metric_faults and trace_anomalies for detailed investigation.
- Correlate these results with logs and traces around earliest timestamps (mostly ~2021-03-06 04:30–04:35 CST) to find root causes (network/TCP, JVM heap, process/zabbix, or application errors).

The original code execution output of IPython Kernel is also provided below for reference:

component  metric_fault_count earliest_metric_fault_start  max_metric_streak  worst_max_deviation_pct  trace_anomaly_count earliest_trace_anomaly_ts
0            MG02                 594     2021-03-06 04:30:00 CST                 31               100.000000                  101   2021-03-06 04:32:00 CST
1            MG01                 571     2021-03-06 04:30:00 CST                 31                82.248866                   98   2021-03-06 04:35:00 CST
2            IG01                 693     2021-03-06 04:30:00 CST                 31               233.333333                   87   2021-03-06 04:32:00 CST
3        Tomcat01                 629     2021-03-06 04:30:00 CST                 31               100.000000                   64   2021-03-06 04:33:00 CST
4            IG02                 721     2021-03-06 04:30:00 CST                 31               233.333333                   63   2021-03-06 04:33:00 CST
5        Tomcat02                 629     2021-03-06 04:30:00 CST                 31               100.000000                   61   2021-03-06 04:33:00 CST
6        dockerB2                  20     2021-03-06 04:36:00 CST                  1                47.826246                   60   2021-03-06 04:33:00 CST
7        Tomcat04                 634     2021-03-06 04:30:00 CST                 31               100.000000                   52   2021-03-06 04:32:00 CST
8        dockerB1                  16     2021-03-06 04:34:00 CST                  1                49.482385                   48   2021-03-06 04:32:00 CST
9        Tomcat03                 635     2021-03-06 04:30:00 CST                 31               133.630924                   47   2021-03-06 04:35:00 CST
10       dockerA1                   6     2021-03-06 04:58:00 CST                  1                 0.000000                   31   2021-03-06 04:35:00 CST
11       dockerA2                  49     2021-03-06 04:34:00 CST                  1                 3.185485                   24   2021-03-06 04:35:00 CST
12        Mysql01                2053     2021-03-06 04:30:00 CST                 31              1900.000000                    0                       NaN
13        Mysql02                1835     2021-03-06 04:30:00 CST                 31              1900.000000                    0                       NaN
14        Redis01                 775     2021-03-06 04:30:00 CST                 31                53.846154                    0                       NaN
15        Redis02                 687     2021-03-06 04:30:00 CST                 31               233.333333                    0                       NaN
16       apache02                 624     2021-03-06 04:30:00 CST                 31               233.333333                    0                       NaN
17       apache01                 157     2021-03-06 04:30:00 CST                 31               146.713792                    0                       NaN
18  ServiceTest10                  12     2021-03-06 04:33:00 CST                  4                64.102487                    0                       NaN
19   ServiceTest6                  12     2021-03-06 04:37:00 CST                  4                17.053708                    0                       NaN```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# reuse existing dataframes if present, otherwise load
if 'log_df' not in globals():
    log_df = pd.read_csv('log.csv')
if 'error_df' not in globals():
    error_df = pd.read_csv('error_logs.csv')

# ensure numeric value column where present
if 'value' in log_df.columns:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# ensure dt columns (timezone-aware Asia/Shanghai)
def ensure_dt(df):
    if 'dt' in df.columns:
        # try to normalize existing dt to tz-aware
        if pd.api.types.is_numeric_dtype(df['dt']):
            df['dt'] = pd.to_datetime(df['dt'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            try:
                if df['dt'].dt.tz is None:
                    df['dt'] = df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
                else:
                    df['dt'] = df['dt'].dt.tz_convert(tz)
            except Exception:
                df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # create from timestamp (seconds)
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

log_df = ensure_dt(log_df)
error_df = ensure_dt(error_df)

# Incident window (inclusive) in Asia/Shanghai
start_ts = tz.localize(pd.Timestamp('2021-03-06 04:30:00'))
end_ts = tz.localize(pd.Timestamp('2021-03-06 05:00:00'))

# 1) Filter log.csv to incident window and store
logs_incident_window = log_df[(log_df['dt'] >= start_ts) & (log_df['dt'] <= end_ts)].copy()

# Aggregate by cmdb_id
if not logs_incident_window.empty:
    # total rows per cmdb_id
    total_rows = logs_incident_window.groupby('cmdb_id').size().reset_index(name='total_log_rows')
    # distinct log_name count
    distinct_names = logs_incident_window.groupby('cmdb_id')['log_name'].nunique().reset_index(name='distinct_log_name_count')
    # earliest timestamp per cmdb_id
    earliest_ts = logs_incident_window.groupby('cmdb_id')['dt'].min().reset_index(name='earliest_dt')
    # top log_name per cmdb_id
    top_names = (
        logs_incident_window.groupby(['cmdb_id', 'log_name'])
        .size()
        .reset_index(name='cnt')
        .sort_values(['cmdb_id', 'cnt'], ascending=[True, False])
    )
    top_per_cmdb = top_names.groupby('cmdb_id').first().reset_index().rename(columns={'log_name':'top_log_name','cnt':'top_log_name_count'})

    # join together
    log_agg = total_rows.merge(distinct_names, on='cmdb_id', how='left')\
                        .merge(top_per_cmdb[['cmdb_id','top_log_name','top_log_name_count']], on='cmdb_id', how='left')\
                        .merge(earliest_ts, on='cmdb_id', how='left')

    log_agg['earliest_log_ts'] = log_agg['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    log_agg = log_agg.drop(columns=['earliest_dt'])

    log_agg_top20 = log_agg.sort_values('total_log_rows', ascending=False).head(20).reset_index(drop=True)
else:
    log_agg_top20 = pd.DataFrame(columns=['cmdb_id','total_log_rows','distinct_log_name_count','top_log_name','top_log_name_count','earliest_log_ts'])

# 2) Filter error_logs.csv to the same window and produce distinct cmdb_id counts and up to 20 samples
error_logs_incident_window = error_df[(error_df['dt'] >= start_ts) & (error_df['dt'] <= end_ts)].copy()

if not error_logs_incident_window.empty:
    error_counts = error_logs_incident_window.groupby('cmdb_id').size().reset_index(name='error_row_count').sort_values('error_row_count', ascending=False)
    # up to 20 sample rows (timestamp, cmdb_id, message) sorted by timestamp asc
    sample_cols = ['timestamp', 'dt', 'cmdb_id', 'message'] if 'message' in error_logs_incident_window.columns else ['timestamp', 'dt', 'cmdb_id']
    error_samples = error_logs_incident_window.sort_values('timestamp')[sample_cols].head(20).copy()
    # format dt for samples
    if 'dt' in error_samples.columns:
        error_samples['dt'] = error_samples['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_counts = pd.DataFrame(columns=['cmdb_id','error_row_count'])
    error_samples = pd.DataFrame(columns=['timestamp','dt','cmdb_id','message']) if 'message' in error_df.columns else pd.DataFrame(columns=['timestamp','dt','cmdb_id'])
    # explicit empty indicator
    error_empty_indicator = "ERROR_LOGS_EMPTY"

# 3) Keyword scan in logs_incident_window over log_name and message if present
keywords = ["OOM","OutOfMemory","error","exception","socket","tcp","CLOSE-WAIT","connection","timeout","heap","gc"]
# prepare a text column combining log_name and message if message exists
if not logs_incident_window.empty:
    if 'message' in logs_incident_window.columns:
        logs_incident_window['_text_search'] = logs_incident_window['log_name'].astype(str) + " " + logs_incident_window['message'].astype(str)
    else:
        logs_incident_window['_text_search'] = logs_incident_window['log_name'].astype(str)

    keyword_counts = []
    text_series = logs_incident_window['_text_search'].astype(str).str.lower()
    for kw in keywords:
        # case-insensitive search
        kw_lower = kw.lower()
        # treat special characters by simple substring match
        mask = text_series.str.contains(re.escape(kw_lower), na=False)
        cnt = int(mask.sum())
        if cnt > 0:
            keyword_counts.append({'keyword': kw, 'count': cnt})
    if keyword_counts:
        keyword_counts_df = pd.DataFrame(keyword_counts).sort_values('count', ascending=False).reset_index(drop=True)
    else:
        keyword_counts_df = pd.DataFrame(columns=['keyword','count'])
    # drop helper column to keep variable compact (but keep in memory as per rules? remove to be safe)
    logs_incident_window = logs_incident_window.drop(columns=['_text_search'])
else:
    keyword_counts_df = pd.DataFrame(columns=['keyword','count'])

# Final compact outputs (store variables already set)
# logs_incident_window, error_logs_incident_window, log_agg_top20, error_counts, error_samples (or error_empty_indicator), keyword_counts_df

# Display compact summaries (limited to 20 rows)
# If error samples empty, show explicit indicator instead of empty df
outputs = (log_agg_top20.head(20),
           error_counts.head(20),
           error_samples.head(20) if not error_logs_incident_window.empty else pd.DataFrame([{'error_logs_incident_window':'EMPTY'}]),
           keyword_counts_df.head(20),
           logs_incident_window.shape,
           error_logs_incident_window.shape)

outputs

```
Out[6]:
```
Summary of bounded log analysis (window 2021-03-06 04:30–05:00 Asia/Shanghai)

1) Aggregated log.csv (filtered to window)
- Stored filtered logs in variable: logs_incident_window (shape = 250 rows).
- Top cmdb_id by total_log_rows (top entries, ordered by total rows):
  - Tomcat01: total_log_rows=52, distinct_log_name_count=3, top_log_name=log.error_count (25), earliest_log_ts=2021-03-06 04:31:00 CST
  - apache02: 50 rows, 2 distinct names, top_log_name=log.error_count (25), earliest 04:32:00 CST
  - Tomcat04: 50 rows, 3 distinct names, top_log_name=log.error_count (24), earliest 04:32:00 CST
  - Tomcat03: 48 rows, 3 distinct names, top_log_name=log.error_count (23), earliest 04:33:00 CST
  - Tomcat02: 46 rows, 3 distinct names, top_log_name=log.error_count (22), earliest 04:33:00 CST
  - Mysql01: 4 rows, 2 distinct names, top_log_name=log.error_count (2), earliest 04:39:00 CST

2) error_logs.csv (filtered to window)
- Stored filtered error logs in variable: error_logs_incident_window (shape = 0 rows).
- No error-log rows were found in the incident window (explicitly empty).

3) Keyword scan in log rows (log_name or message)
- Keywords found and counts (only keywords with count>0 shown):
  - "error": 121 occurrences
  - "gc": 8 occurrences
- Other keywords from the scan list (OOM, OutOfMemory, exception, socket, tcp, CLOSE-WAIT, connection, timeout, heap) were not found in the filtered log fields (count = 0).

Quick interpretation / next steps
- Tomcat and apache hosts dominate logs in the window, and log.error_count is the most frequent log_name — investigate application errors on Tomcat01/02/03/04 and apache02 around 04:30–04:33.
- No explicit error_log rows in error_logs.csv for the window; rely on log.csv rows and traces/metrics for root-cause correlation.
- The high "error" keyword count and some "gc" mentions suggest application errors and occasional GC activity worth correlating with JVM/heap metrics and trace anomalies for the same hosts/timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_log_rows  distinct_log_name_count     top_log_name  top_log_name_count          earliest_log_ts
0  Tomcat01              52                        3  log.error_count                  25  2021-03-06 04:31:00 CST
1  apache02              50                        2  log.error_count                  25  2021-03-06 04:32:00 CST
2  Tomcat04              50                        3  log.error_count                  24  2021-03-06 04:32:00 CST
3  Tomcat03              48                        3  log.error_count                  23  2021-03-06 04:33:00 CST
4  Tomcat02              46                        3  log.error_count                  22  2021-03-06 04:33:00 CST
5   Mysql01               4                        2  log.error_count                   2  2021-03-06 04:39:00 CST, Empty DataFrame
Columns: [cmdb_id, error_row_count]
Index: [],   error_logs_incident_window
0                      EMPTY,   keyword  count
0   error    121
1      gc      8, (250, 5), (0, 4))```
```

In [ ]:
# In[7]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required kernel variables exist
if 'metric_faults' not in globals():
    raise RuntimeError("metric_faults not found in kernel.")
if 'trace_anomalies' not in globals():
    raise RuntimeError("trace_anomalies not found in kernel.")
if 'logs_incident_window' not in globals():
    raise RuntimeError("logs_incident_window not found in kernel.")

# Reuse variables
mf = metric_faults.copy()
ta = trace_anomalies.copy()
logs_win = logs_incident_window.copy()

# Ensure datetime columns are timezone-aware Asia/Shanghai
def ensure_dt_col(df, col):
    if col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_datetime(df[col], unit='s', utc=True).dt.tz_convert(tz)
        else:
            try:
                if df[col].dt.tz is None:
                    df[col] = df[col].dt.tz_localize('UTC').dt.tz_convert(tz)
                else:
                    df[col] = df[col].dt.tz_convert(tz)
            except Exception:
                df[col] = pd.to_datetime(df[col], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df[col] = pd.NaT
    return df

mf = ensure_dt_col(mf, 'start_dt')
ta = ensure_dt_col(ta, 'dt')
logs_win = ensure_dt_col(logs_win, 'dt')

# 1) Extract trace component token if not already present
if 'trace_component' not in ta.columns:
    def extract_token(r):
        name = r.get('trace_name', '')
        cm = r.get('cmdb_id', pd.NA)
        if not isinstance(name, str):
            return ''
        m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)\b', name)
        if m:
            return m.group(1)
        if 'self' in name:
            return cm if pd.notna(cm) else 'self'
        return ''
    ta['trace_component'] = ta.apply(extract_token, axis=1)

# List of candidate components requested
candidates = [
    "IG01","IG02","MG01","MG02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "Mysql01","Mysql02","Redis01","Redis02",
    "apache01","apache02"
]

# 2) Aggregate metric_faults by component (cmdb_id)
metric_agg = mf.groupby('cmdb_id').agg(
    metric_fault_count=('kpi_name', 'count'),
    earliest_metric_fault_start_dt=('start_dt', 'min'),
    max_metric_streak=('streak_length', 'max'),
    worst_max_deviation_pct=('max_deviation_pct', 'max')
).reset_index().rename(columns={'cmdb_id':'component'})

# Format earliest_metric_fault_start as string (UTC+8)
metric_agg['earliest_metric_fault_start'] = metric_agg['earliest_metric_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_agg.loc[metric_agg['earliest_metric_fault_start_dt'].isna(), 'earliest_metric_fault_start'] = pd.NA
metric_agg = metric_agg.drop(columns=['earliest_metric_fault_start_dt'])

# 3) Aggregate trace_anomalies by extracted trace_component
trace_agg = ta[ta['trace_component'].notna() & (ta['trace_component'] != '')].groupby('trace_component').agg(
    trace_anomaly_count=('trace_component', 'count'),
    earliest_trace_anomaly_dt=('dt', 'min')
).reset_index().rename(columns={'trace_component':'component'})

trace_agg['earliest_trace_anomaly_ts'] = trace_agg['earliest_trace_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_agg.loc[trace_agg['earliest_trace_anomaly_dt'].isna(), 'earliest_trace_anomaly_ts'] = pd.NA
trace_agg = trace_agg.drop(columns=['earliest_trace_anomaly_dt'])

# 4) Aggregate logs within window by cmdb_id
if not logs_win.empty:
    log_agg = logs_win.groupby('cmdb_id').agg(
        total_log_rows_in_window=('cmdb_id', 'size'),
        earliest_log_dt=('dt', 'min')
    ).reset_index().rename(columns={'cmdb_id':'component'})
    log_agg['earliest_log_ts_in_window'] = log_agg['earliest_log_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    log_agg.loc[log_agg['earliest_log_dt'].isna(), 'earliest_log_ts_in_window'] = pd.NA
    log_agg = log_agg.drop(columns=['earliest_log_dt'])
else:
    log_agg = pd.DataFrame(columns=['component','total_log_rows_in_window','earliest_log_ts_in_window'])

# 5) Merge for candidate components
cand_df = pd.DataFrame({'component': candidates})

joined = cand_df.merge(metric_agg, on='component', how='left')\
                .merge(trace_agg, on='component', how='left')\
                .merge(log_agg, on='component', how='left')

# Ensure numeric columns and fill missing with NaN/NULL behavior
joined['metric_fault_count'] = joined['metric_fault_count'].where(joined['metric_fault_count'].notna(), pd.NA)
joined['max_metric_streak'] = joined['max_metric_streak'].where(joined['max_metric_streak'].notna(), pd.NA)
joined['worst_max_deviation_pct'] = joined['worst_max_deviation_pct'].where(joined['worst_max_deviation_pct'].notna(), pd.NA)
joined['trace_anomaly_count'] = joined['trace_anomaly_count'].where(joined['trace_anomaly_count'].notna(), pd.NA)
joined['total_log_rows_in_window'] = joined['total_log_rows_in_window'].where(joined['total_log_rows_in_window'].notna(), pd.NA)

# 6) Order rows by earliest_metric_fault_start asc (NaT/NaN last) then worst_max_deviation_pct desc
# For sorting, convert earliest_metric_fault_start to datetime for ordering, handling NaNs
joined['__earliest_m_dt'] = pd.to_datetime(joined['earliest_metric_fault_start'], format='%Y-%m-%d %H:%M:%S %Z', errors='coerce')
joined = joined.sort_values(by=['__earliest_m_dt','worst_max_deviation_pct'], ascending=[True, False], na_position='last').drop(columns=['__earliest_m_dt'])

# 7) Select and order final columns exactly as requested
candidate_timeline = joined[[
    'component',
    'metric_fault_count',
    'earliest_metric_fault_start',
    'max_metric_streak',
    'worst_max_deviation_pct',
    'trace_anomaly_count',
    'earliest_trace_anomaly_ts',
    'total_log_rows_in_window',
    'earliest_log_ts_in_window'
]].head(20).reset_index(drop=True)

# Display the compact table (and store in kernel variable candidate_timeline)
candidate_timeline

```
Out[7]:
```
Compact timeline summary for candidate components (window 2021-03-06 04:30–05:00 Asia/Shanghai):

Overview
- Most components show persistent metric faults starting at 2021-03-06 04:30:00 CST with max_metric_streak = 31 minutes (i.e., continuous anomalies across the whole window).
- Key components with both many metric faults and trace activity:
  - IG01: 693 metric faults, earliest fault 2021-03-06 04:30, max streak 31, worst deviation ~233%, 87 trace anomalies (earliest trace 04:32).
  - IG02: 721 metric faults, earliest 04:30, max streak 31, worst deviation ~233%, 63 trace anomalies (earliest 04:33).
  - MG02: 594 metric faults, earliest 04:30, max streak 31, worst deviation 100%, 101 trace anomalies (earliest 04:32).
  - MG01: 571 metric faults, earliest 04:30, max streak 31, worst deviation ~82%, 98 trace anomalies (earliest 04:35).
  - Tomcat01/02/03/04: each ~629–635 metric faults, earliest 04:30, max streak 31, worst deviations ~100% (Tomcat01/02/04) and ~133% (Tomcat03). Trace anomalies: Tomcat01=64 (04:33), Tomcat02=61 (04:33), Tomcat03=47 (04:35), Tomcat04=52 (04:32). Log rows in window: Tomcat01=52 (earliest log 04:31), Tomcat02=46 (04:33), Tomcat03=48 (04:33), Tomcat04=50 (04:32).
- Database / infra:
  - Mysql01: very large metric fault count (2053), earliest 04:30, max streak 31, huge worst deviation (1900%), but ZERO trace anomalies in the extracted trace_anomalies. There are a few log rows for Mysql01 in the window (4 rows, earliest 04:39).
  - Mysql02: 1835 metric faults, similar profile to Mysql01 (no trace anomalies, no logs in window).
  - Redis01/Redis02: many metric faults (Redis01=775, Redis02=687), max streak 31; Redis01/02 show no trace anomalies recorded in the trace_anomalies variable for the window and no log rows in window.
  - apache02: 624 metric faults, worst deviation ~233%, no trace anomalies, but 50 log rows in window (earliest 04:32). apache01 also shows many metric faults (157) but no traces/logs in window.
- Missing/NULL values:
  - Several components (e.g., Mysql02, Redis*, apache01) have no trace_anomaly_count or no log rows in the incident-window datasets (values shown as NaN/NULL).

Actionable interpretation
- IG and MG nodes show strong correlation between persistent metric faults and trace anomalies (start ~04:30–04:35). These should be prioritized for end-to-end tracing and application/cluster investigation.
- Tomcat nodes show correlated metric faults, trace anomalies, and substantial log activity (log.error_count present) — investigate application errors and request handling between 04:30–04:35.
- Mysql and Redis servers show very large, persistent metric faults but no matching trace anomalies in the extracted traces — investigate infra-level issues (DB performance, connection/socket states, process/resource issues) and check other logs or monitoring sources since traces didn’t capture them in this window.
- Components with logs present in-window (Tomcat and apache nodes) should be examined first for error logs around the earliest timestamps indicated (04:31–04:33).

Data for follow-up
- The compact table is stored in kernel variable candidate_timeline for inspection and triage. Use metric_faults, trace_anomalies, and logs_incident_window to drill into specific KPIs, spans, and log messages around the earliest timestamps noted above.

The original code execution output of IPython Kernel is also provided below for reference:

component  metric_fault_count earliest_metric_fault_start  max_metric_streak  worst_max_deviation_pct  trace_anomaly_count earliest_trace_anomaly_ts  total_log_rows_in_window earliest_log_ts_in_window
0    Mysql01                2053     2021-03-06 04:30:00 CST                 31              1900.000000                  NaN                       NaN                       4.0   2021-03-06 04:39:00 CST
1    Mysql02                1835     2021-03-06 04:30:00 CST                 31              1900.000000                  NaN                       NaN                       NaN                       NaN
2    Redis02                 687     2021-03-06 04:30:00 CST                 31               233.333333                  NaN                       NaN                       NaN                       NaN
3       IG01                 693     2021-03-06 04:30:00 CST                 31               233.333333                 87.0   2021-03-06 04:32:00 CST                       NaN                       NaN
4       IG02                 721     2021-03-06 04:30:00 CST                 31               233.333333                 63.0   2021-03-06 04:33:00 CST                       NaN                       NaN
5   apache02                 624     2021-03-06 04:30:00 CST                 31               233.333333                  NaN                       NaN                      50.0   2021-03-06 04:32:00 CST
6   apache01                 157     2021-03-06 04:30:00 CST                 31               146.713792                  NaN                       NaN                       NaN                       NaN
7   Tomcat03                 635     2021-03-06 04:30:00 CST                 31               133.630924                 47.0   2021-03-06 04:35:00 CST                      48.0   2021-03-06 04:33:00 CST
8       MG02                 594     2021-03-06 04:30:00 CST                 31               100.000000                101.0   2021-03-06 04:32:00 CST                       NaN                       NaN
9   Tomcat01                 629     2021-03-06 04:30:00 CST                 31               100.000000                 64.0   2021-03-06 04:33:00 CST                      52.0   2021-03-06 04:31:00 CST
10  Tomcat02                 629     2021-03-06 04:30:00 CST                 31               100.000000                 61.0   2021-03-06 04:33:00 CST                      46.0   2021-03-06 04:33:00 CST
11  Tomcat04                 634     2021-03-06 04:30:00 CST                 31               100.000000                 52.0   2021-03-06 04:32:00 CST                      50.0   2021-03-06 04:32:00 CST
12      MG01                 571     2021-03-06 04:30:00 CST                 31                82.248866                 98.0   2021-03-06 04:35:00 CST                       NaN                       NaN
13   Redis01                 775     2021-03-06 04:30:00 CST                 31                53.846154                  NaN                       NaN                       NaN                       NaN```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables
mf = metric_faults  # faults table from previous steps
ma = metric_anomalies  # anomaly rows

# Focus components
components = ['Mysql01', 'Mysql02']

# Helper: produce formatted start/end timestamp columns in mf
def ensure_ts_format(df):
    df = df.copy()
    # start
    if 'start_ts_str' in df.columns:
        df['start_ts_fmt'] = df['start_ts_str']
    elif 'start_dt' in df.columns:
        # ensure tz-aware
        df['start_dt'] = pd.to_datetime(df['start_dt'])
        try:
            df['start_ts_fmt'] = df['start_dt'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            df['start_ts_fmt'] = df['start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
    elif 'start_timestamp' in df.columns:
        df['start_ts_fmt'] = pd.to_datetime(df['start_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        df['start_ts_fmt'] = pd.NA

    # end
    if 'end_ts_str' in df.columns:
        df['end_ts_fmt'] = df['end_ts_str']
    elif 'end_dt' in df.columns:
        df['end_dt'] = pd.to_datetime(df['end_dt'])
        try:
            df['end_ts_fmt'] = df['end_dt'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            df['end_ts_fmt'] = df['end_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
    elif 'end_timestamp' in df.columns:
        df['end_ts_fmt'] = pd.to_datetime(df['end_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        df['end_ts_fmt'] = pd.NA

    return df

mf_fmt = ensure_ts_format(mf)

# 1) Top 5 faults per component ordered by worst_max_deviation_pct desc
top_faults_list = []
for comp in components:
    comp_faults = mf_fmt[mf_fmt['cmdb_id'] == comp].copy()
    if comp_faults.empty:
        continue
    # sort by max_deviation_pct desc, NaNs to bottom
    comp_top = comp_faults.sort_values('max_deviation_pct', ascending=False, na_position='last').head(5)
    # select required columns and rename timestamps
    comp_top_sel = comp_top[[
        'cmdb_id',
        'kpi_name',
        'start_ts_fmt',
        'end_ts_fmt',
        'streak_length',
        'threshold_type',
        'threshold_value',
        'max_deviation_pct',
        'max_value'
    ]].rename(columns={
        'start_ts_fmt': 'start_timestamp',
        'end_ts_fmt': 'end_timestamp'
    })
    top_faults_list.append(comp_top_sel)

# Concatenate results (will have up to 10 rows: 5 per component)
if top_faults_list:
    top5_faults_table = pd.concat(top_faults_list, ignore_index=True)
else:
    top5_faults_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','start_timestamp','end_timestamp','streak_length',
        'threshold_type','threshold_value','max_deviation_pct','max_value'
    ])

# 2) For the single top fault per component (highest max_deviation_pct), retrieve earliest matching anomalous row from metric_anomalies
earliest_anom_rows = []
# Ensure metric_anomalies dt formatting
if 'dt' in ma.columns:
    ma = ma.copy()
    ma['dt'] = pd.to_datetime(ma['dt'])
    try:
        ma['dt_fmt'] = ma['dt'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        ma['dt_fmt'] = ma['dt'].dt.strftime('%Y-%m-%d %H:%M:%S')

for comp in components:
    comp_faults = mf_fmt[mf_fmt['cmdb_id'] == comp].copy()
    if comp_faults.empty:
        continue
    # pick top fault by max_deviation_pct
    top_fault = comp_faults.sort_values('max_deviation_pct', ascending=False, na_position='last').head(1)
    if top_fault.empty:
        continue
    top_kpi = top_fault.iloc[0]['kpi_name']
    top_threshold_type = top_fault.iloc[0]['threshold_type']
    top_threshold_value = top_fault.iloc[0]['threshold_value']

    # find earliest matching anomalous row in metric_anomalies for this component and kpi_name
    matched = ma[(ma['cmdb_id'] == comp) & (ma['kpi_name'] == top_kpi)].copy()
    if matched.empty:
        continue
    matched_sorted = matched.sort_values('timestamp', ascending=True)
    earliest = matched_sorted.iloc[0]
    # Compose output row with requested columns: timestamp, cmdb_id, kpi_name, value, joined threshold_value, threshold_type
    # timestamp use dt_fmt if present else convert from timestamp
    ts = earliest.get('dt_fmt') if 'dt_fmt' in matched_sorted.columns else pd.to_datetime(earliest['timestamp'], unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    row = {
        'timestamp': ts,
        'cmdb_id': earliest.get('cmdb_id'),
        'kpi_name': earliest.get('kpi_name'),
        'value': earliest.get('value'),
        'threshold_value': top_threshold_value,
        'threshold_type': top_threshold_type
    }
    earliest_anom_rows.append(row)

# Create DataFrame for earliest anomalous rows
if earliest_anom_rows:
    earliest_anom_rows = pd.DataFrame(earliest_anom_rows)[[
        'timestamp','cmdb_id','kpi_name','value','threshold_value','threshold_type'
    ]]
else:
    earliest_anom_rows = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','threshold_value','threshold_type'])

# Final outputs (compact)
top5_faults_table, earliest_anom_rows

```
Out[8]:
```
summary = (
    "Focused inspection — Mysql01 & Mysql02 (compact):\n\n"
    "Mysql01 — top 5 fault sequences (ordered by max_deviation_pct):\n"
    "1) Mysql-MySQL_3306_SlowQueries | 2021-03-06 04:50 → 2021-03-06 04:51 | streak=2 | type=high | threshold=0.000835 | max_deviation_pct=1900.0% | max_value=0.0167\n"
    "2) Mysql-MySQL_3306_Com Insert  | 2021-03-06 04:30 → 2021-03-06 04:31 | streak=2 | type=high | threshold=0.013335 | max_deviation_pct=1900.0% | max_value=0.2667\n"
    "3) Mysql-MySQL_3306_Innodb log write requests | 2021-03-06 04:30 → 2021-03-06 04:31 | streak=2 | type=high | threshold=0.160245 | max_deviation_pct=1096.105963% | max_value=1.9167\n"
    "4) Mysql-MySQL_3306_Innodb buffer pool pages dirty | 2021-03-06 04:30 → 2021-03-06 04:31 | streak=2 | type=high | threshold=8.05 | max_deviation_pct=955.900621% | max_value=85.0000\n"
    "5) Mysql-MySQL_3306_Innodb buffer pool write requests | 2021-03-06 04:30 → 2021-03-06 04:31 | streak=2 | type=high | threshold=0.804135 | max_deviation_pct=521.786143% | max_value=5.0000\n\n"
    "Mysql02 — top 5 fault sequences (ordered by max_deviation_pct):\n"
    "1) Mysql-MySQL_3306_Innodb os log pending fsyncs | 2021-03-06 04:53 → 2021-03-06 04:53 | streak=1 | type=high | threshold=0.05 | max_deviation_pct=1900.0% | max_value=1.0\n"
    "2) Mysql-MySQL_3306_Innodb data pending fsyncs   | 2021-03-06 04:53 → 2021-03-06 04:53 | streak=1 | type=high | threshold=0.05 | max_deviation_pct=1900.0% | max_value=1.0\n"
    "3) Mysql-MySQL_3306_Handler Read Rnd             | 2021-03-06 04:39 → 2021-03-06 04:39 | streak=1 | type=high | threshold=0.179165 | max_deviation_pct=309.287528% | max_value=0.7333\n"
    "4) Mysql-MySQL_3306_Sort Range                   | 2021-03-06 04:39 → 2021-03-06 04:39 | streak=1 | type=high | threshold=0.179165 | max_deviation_pct=309.287528% | max_value=0.7333\n"
    "5) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps| 2021-03-06 04:56 → 2021-03-06 04:56 | streak=1 | type=high | threshold=0.3 | max_deviation_pct=233.333333% | max_value=1.0\n\n"
    "Earliest matching anomalous row for the single top fault (highest max_deviation_pct) per component:\n"
    "- Mysql01: timestamp=2021-03-06 04:30:00 CST | kpi=Mysql-MySQL_3306_SlowQueries | value=0.0 | threshold_value=0.000835 | threshold_type=high\n"
    "- Mysql02: timestamp=2021-03-06 04:30:00 CST | kpi=Mysql-MySQL_3306_Innodb os log pending fsyncs | value=0.0 | threshold_value=0.05 | threshold_type=high\n\n"
    "Notes / interpretation:\n"
    "- Top faults for both DBs are short sequences (mostly 1–2 minutes) but show very large reported deviations vs the global thresholds.\n"
    "- The earliest anomalous rows for the selected top KPIs show value=0.0 while threshold_value>0 and threshold_type='high' — this indicates the anomaly flagging logic (using global P95/P05) produced matches that should be reviewed (possible threshold smallness or data normalization effects).\n"
    "- Recommended next steps: inspect raw KPI timeseries for these specific KPIs around the listed timestamps, correlate with DB logs and system I/O (fsyncs, buffer pool activity), and verify whether thresholds/P95 values are reasonable for these normalized metrics."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name          start_timestamp            end_timestamp  streak_length threshold_type  threshold_value  max_deviation_pct  max_value
0  Mysql01                       Mysql-MySQL_3306_SlowQueries  2021-03-06 04:50:00 CST  2021-03-06 04:51:00 CST              2           high         0.000835        1900.000000     0.0167
1  Mysql01                        Mysql-MySQL_3306_Com Insert  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.013335        1900.000000     0.2667
2  Mysql01         Mysql-MySQL_3306_Innodb log write requests  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.160245        1096.105963     1.9167
3  Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         8.050000         955.900621    85.0000
4  Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.804135         521.786143     5.0000
5  Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs  2021-03-06 04:53:00 CST  2021-03-06 04:53:00 CST              1           high         0.050000        1900.000000     1.0000
6  Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs  2021-03-06 04:53:00 CST  2021-03-06 04:53:00 CST              1           high         0.050000        1900.000000     1.0000
7  Mysql02                  Mysql-MySQL_3306_Handler Read Rnd  2021-03-06 04:39:00 CST  2021-03-06 04:39:00 CST              1           high         0.179165         309.287528     0.7333
8  Mysql02                        Mysql-MySQL_3306_Sort Range  2021-03-06 04:39:00 CST  2021-03-06 04:39:00 CST              1           high         0.179165         309.287528     0.7333
9  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-06 04:56:00 CST  2021-03-06 04:56:00 CST              1           high         0.300000         233.333333     1.0000,                  timestamp  cmdb_id                                       kpi_name  value  threshold_value threshold_type
0  2021-03-06 04:30:00 CST  Mysql01                   Mysql-MySQL_3306_SlowQueries    0.0         0.000835           high
1  2021-03-06 04:30:00 CST  Mysql02  Mysql-MySQL_3306_Innodb os log pending fsyncs    0.0         0.050000           high)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing metric_faults variable
df = metric_faults.copy()

# Ensure start/end timestamp strings in Asia/Shanghai
if 'start_ts_str' in df.columns:
    df['start_timestamp'] = df['start_ts_str']
elif 'start_dt' in df.columns:
    df['start_timestamp'] = pd.to_datetime(df['start_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
elif 'start_timestamp' in df.columns:
    df['start_timestamp'] = pd.to_datetime(df['start_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    df['start_timestamp'] = pd.NA

if 'end_ts_str' in df.columns:
    df['end_timestamp'] = df['end_ts_str']
elif 'end_dt' in df.columns:
    df['end_timestamp'] = pd.to_datetime(df['end_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
elif 'end_timestamp' in df.columns:
    df['end_timestamp'] = pd.to_datetime(df['end_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    df['end_timestamp'] = pd.NA

# Select and coerce numeric columns
for col in ['threshold_value','max_value','max_deviation_pct']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = pd.NA

# Prepare final table: top 10 by max_deviation_pct desc
top10_faults = df.sort_values(by='max_deviation_pct', ascending=False, na_position='last').head(10)[[
    'cmdb_id',
    'kpi_name',
    'start_timestamp',
    'end_timestamp',
    'streak_length',
    'threshold_type',
    'threshold_value',
    'max_value',
    'max_deviation_pct'
]].reset_index(drop=True)

top10_faults

```
Out[9]:
```
Top 10 fault sequences by max_deviation_pct (columns shown: cmdb_id, kpi_name, start → end, streak_length, threshold_type, threshold_value, max_value, max_deviation_pct). Times are Asia/Shanghai (CST).

1. Mysql02 — Mysql-MySQL_3306_Innodb data pending fsyncs — 2021-03-06 04:53:00 → 04:53:00 | streak=1 | high | threshold=0.050000 | max_value=1.0000 | max_deviation_pct=1900.0  
2. Mysql02 — Mysql-MySQL_3306_Innodb os log pending fsyncs — 2021-03-06 04:53:00 → 04:53:00 | streak=1 | high | threshold=0.050000 | max_value=1.0000 | max_deviation_pct=1900.0  
3. Mysql01 — Mysql-MySQL_3306_Com Insert — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.013335 | max_value=0.2667 | max_deviation_pct=1900.0  
4. Mysql01 — Mysql-MySQL_3306_SlowQueries — 2021-03-06 04:50:00 → 04:51:00 | streak=2 | high | threshold=0.000835 | max_value=0.0167 | max_deviation_pct=1900.0  
5. Mysql01 — Mysql-MySQL_3306_Innodb log write requests — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.160245 | max_value=1.9167 | max_deviation_pct=1096.106  
6. Mysql01 — Mysql-MySQL_3306_Innodb buffer pool pages dirty — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=8.05 | max_value=85.0000 | max_deviation_pct=955.901  
7. Mysql01 — Mysql-MySQL_3306_Innodb buffer pool write requests — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.804135 | max_value=5.0000 | max_deviation_pct=521.786  
8. Mysql01 — Mysql-MySQL_3306_Innodb dblwr pages written — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.324200 | max_value=1.4167 | max_deviation_pct=336.983  
9. Mysql01 — Mysql-MySQL_3306_Innodb buffer pool pages flushed — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.324200 | max_value=1.4167 | max_deviation_pct=336.983  
10. Mysql01 — Mysql-MySQL_3306_Innodb pages written — 2021-03-06 04:30:00 → 04:31:00 | streak=2 | high | threshold=0.324200 | max_value=1.4167 | max_deviation_pct=336.983

Conclusion: the maximum observed deviation = 1900.0%. That value occurs for multiple fault sequences (Mysql02: two InnoDB fsync-related KPIs at 2021-03-06 04:53:00; and Mysql01: Com Insert at 04:30 and SlowQueries at 04:50). The earliest fault among those with the top deviation is Mysql01 — Mysql-MySQL_3306_Com Insert starting at 2021-03-06 04:30:00 CST.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name          start_timestamp            end_timestamp  streak_length threshold_type  threshold_value  max_value  max_deviation_pct
0  Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs  2021-03-06 04:53:00 CST  2021-03-06 04:53:00 CST              1           high         0.050000     1.0000        1900.000000
1  Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs  2021-03-06 04:53:00 CST  2021-03-06 04:53:00 CST              1           high         0.050000     1.0000        1900.000000
2  Mysql01                        Mysql-MySQL_3306_Com Insert  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.013335     0.2667        1900.000000
3  Mysql01                       Mysql-MySQL_3306_SlowQueries  2021-03-06 04:50:00 CST  2021-03-06 04:51:00 CST              2           high         0.000835     0.0167        1900.000000
4  Mysql01         Mysql-MySQL_3306_Innodb log write requests  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.160245     1.9167        1096.105963
5  Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         8.050000    85.0000         955.900621
6  Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.804135     5.0000         521.786143
7  Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.324200     1.4167         336.983344
8  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages flushed  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.324200     1.4167         336.983344
9  Mysql01              Mysql-MySQL_3306_Innodb pages written  2021-03-06 04:30:00 CST  2021-03-06 04:31:00 CST              2           high         0.324200     1.4167         336.983344```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel state; ensure it's loaded
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure dt column is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize existing dt to tz-aware Asia/Shanghai if needed
    if pd.api.types.is_numeric_dtype(metric_df['dt']):
        metric_df['dt'] = pd.to_datetime(metric_df['dt'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        try:
            if metric_df['dt'].dt.tz is None:
                metric_df['dt'] = metric_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
            else:
                metric_df['dt'] = metric_df['dt'].dt.tz_convert(tz)
        except Exception:
            metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target KPIs for Mysql01
target_kpis = [
    'Mysql-MySQL_3306_Com Insert',
    'Mysql-MySQL_3306_Innodb log write requests',
    'Mysql-MySQL_3306_Innodb buffer pool pages dirty',
    'Mysql-MySQL_3306_SlowQueries'
]

# 1) Compute global thresholds (P95 and P05) using the entire metric series (per rule)
thresholds = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric_df for Mysql01 and the target KPIs
start_dt = tz.localize(pd.Timestamp('2021-03-06 04:25:00'))
end_dt = tz.localize(pd.Timestamp('2021-03-06 04:40:00'))

mysql01_rows = metric_df[
    (metric_df['cmdb_id'] == 'Mysql01') &
    (metric_df['kpi_name'].isin(target_kpis)) &
    (metric_df['dt'] >= start_dt) &
    (metric_df['dt'] <= end_dt)
].copy()

# 3) Join global P95/P05 thresholds
mysql01_rows = mysql01_rows.merge(
    thresholds,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Determine threshold_type per row (based on global P95/P05)
def determine_threshold_type(row):
    v = row['value']
    if pd.isna(v):
        return pd.NA
    p95 = row.get('p95', np.nan)
    p05 = row.get('p05', np.nan)
    if pd.notna(p95) and v >= p95:
        return 'high'
    if pd.notna(p05) and v <= p05:
        return 'low'
    return pd.NA

mysql01_rows['threshold_type'] = mysql01_rows.apply(determine_threshold_type, axis=1)

# Format timestamp for display (Asia/Shanghai)
mysql01_rows['timestamp_str'] = mysql01_rows['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order columns as requested, sort by timestamp asc then kpi_name
mysql01_topkpi_timeseries = mysql01_rows.sort_values(['dt', 'kpi_name'], ascending=[True, True])[
    ['timestamp_str', 'cmdb_id', 'kpi_name', 'value', 'p95', 'threshold_type']
].rename(columns={
    'timestamp_str': 'timestamp',
    'p95': 'threshold_value_P95'
}).head(20).reset_index(drop=True)

# Store in kernel variable (already assigned) and return compact table
mysql01_topkpi_timeseries

```
Out[10]:
```
Findings (Mysql01 top KPIs near incident start)

- Earliest anomalous timestamp confirmed: 2021-03-06 04:30:00 CST.
- At 04:30:00 three KPIs were flagged high (value >= global P95):
  - Mysql-MySQL_3306_Com Insert: value=0.2667, P95=0.013335 (high)
  - Mysql-MySQL_3306_Innodb buffer pool pages dirty: value=85.0000, P95=8.050000 (high)
  - Mysql-MySQL_3306_Innodb log write requests: value=1.9167, P95=0.160245 (high)
- The SlowQueries KPI at 04:30:00 had value=0.0 and was marked low (value <= P05) with P95 shown=0.000835 (threshold_type='low').
- Subsequent rows (04:31–04:36) often show 0.0 for these KPIs, producing low-type flags in those minutes.
- The extracted timeseries is stored in kernel variable mysql01_topkpi_timeseries for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  cmdb_id                                         kpi_name    value  threshold_value_P95 threshold_type
0   2021-03-06 04:30:00 CST  Mysql01                      Mysql-MySQL_3306_Com Insert   0.2667             0.013335           high
1   2021-03-06 04:30:00 CST  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty  85.0000             8.050000           high
2   2021-03-06 04:30:00 CST  Mysql01       Mysql-MySQL_3306_Innodb log write requests   1.9167             0.160245           high
3   2021-03-06 04:30:00 CST  Mysql01                     Mysql-MySQL_3306_SlowQueries   0.0000             0.000835            low
4   2021-03-06 04:31:00 CST  Mysql01                      Mysql-MySQL_3306_Com Insert   0.0000             0.013335            low
5   2021-03-06 04:31:00 CST  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty   0.0000             8.050000            low
6   2021-03-06 04:31:00 CST  Mysql01       Mysql-MySQL_3306_Innodb log write requests   0.0000             0.160245            low
7   2021-03-06 04:31:00 CST  Mysql01                     Mysql-MySQL_3306_SlowQueries   0.0000             0.000835            low
8   2021-03-06 04:33:00 CST  Mysql01                      Mysql-MySQL_3306_Com Insert   0.0000             0.013335            low
9   2021-03-06 04:33:00 CST  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty   0.0000             8.050000            low
10  2021-03-06 04:33:00 CST  Mysql01       Mysql-MySQL_3306_Innodb log write requests   0.0000             0.160245            low
11  2021-03-06 04:33:00 CST  Mysql01                     Mysql-MySQL_3306_SlowQueries   0.0000             0.000835            low
12  2021-03-06 04:35:00 CST  Mysql01                      Mysql-MySQL_3306_Com Insert   0.0000             0.013335            low
13  2021-03-06 04:35:00 CST  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty   0.0000             8.050000            low
14  2021-03-06 04:35:00 CST  Mysql01       Mysql-MySQL_3306_Innodb log write requests   0.0000             0.160245            low
15  2021-03-06 04:35:00 CST  Mysql01                     Mysql-MySQL_3306_SlowQueries   0.0000             0.000835            low
16  2021-03-06 04:36:00 CST  Mysql01                      Mysql-MySQL_3306_Com Insert   0.0000             0.013335            low
17  2021-03-06 04:36:00 CST  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty   0.0000             8.050000            low
18  2021-03-06 04:36:00 CST  Mysql01       Mysql-MySQL_3306_Innodb log write requests   0.0000             0.160245            low
19  2021-03-06 04:36:00 CST  Mysql01                     Mysql-MySQL_3306_SlowQueries   0.0000             0.000835            low```
```